# Question 6

In [1]:
#Using smth means: pre-compile and load smth as a package.
using Pkg;
#Set the Gurobi path
ENV["GUROBI_HOME"] = "C:\\gurobi1103\\win64"
#For first timers (comment after): compile Gurobi with the given path
Pkg.build("Gurobi")
#Add the packages that may not be installed.
Pkg.add(["Latexify", "Polyhedra", "Plots", "Makie", "CDDLib", "JuMP", "Gurobi", "AbstractPlotting"])
#Load the packages
using JuMP, Latexify, Gurobi, LinearAlgebra, Polyhedra, CDDLib, Plots, SparseArrays, Makie, AbstractPlotting

display("Initialization done! All set :-)")

    Building Gurobi → `C:\Users\user\.julia\scratchspaces\44cfe95a-1eb2-52ea-b672-e2afdf69b78f\53cc56f49295c6b41da670fcf0e0cd812c9580c7\build.log`
   Resolving package versions...
  No Changes to `C:\Users\user\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\user\.julia\environments\v1.11\Manifest.toml`
┌ Warning: Everything from AbstractPlotting.jl got moved to Makie.jl. 
│ Have a look at the README for information on how to upgrade: 
│ https://github.com/JuliaPlots/AbstractPlotting.jl
└ @ AbstractPlotting C:\Users\user\.julia\packages\AbstractPlotting\2A5iv\src\AbstractPlotting.jl:261


"Initialization done! All set :-)"

In [2]:
Pkg.status("Gurobi")

Status `C:\Users\user\.julia\environments\v1.11\Project.toml`
⌃ [2e9cd046] Gurobi v1.3.1
Info Packages marked with ⌃ have new versions available and may be upgradable.


In [3]:
using Pkg
Pkg.add("CSV")
Pkg.add("DataFrames")
Pkg.add("Plots")

   Resolving package versions...
  No Changes to `C:\Users\user\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\user\.julia\environments\v1.11\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\user\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\user\.julia\environments\v1.11\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\user\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\user\.julia\environments\v1.11\Manifest.toml`


In [4]:
# Model parameters

thetas = [0.4234, 0.5766]

# Updated intercepts (β₀ for Segment 1 and Segment 2)
intercepts = [-1.7391, -1.7551]  # Non-Sat, Sat

# Updated betas: rows = segments, columns = features (β₁ to β₈)
betas = [
    0.473942   0.109629   0.108720   0.016666   0.052382  -0.070328  -1.568860  0.147910;  # Segment 1 (Non-Sat)
    0.358241   0.111053   0.098476   0.025599   0.036260  -0.061769  -1.155128  0.167791   # Segment 2 (Sat)
]

# Datasets

using CSV, DataFrames
# Read CSV file into DataFrame
data1 = CSV.read("data1.csv", DataFrame)
data2 = CSV.read("data2.csv", DataFrame)
data3 = CSV.read("data3.csv", DataFrame)
data4 = CSV.read("data4.csv", DataFrame)
df_norm_params = CSV.read("normalization_params_q1.csv", DataFrame)

data1_original = deepcopy(data1)
data2_original = deepcopy(data2)
data3_original = deepcopy(data3)
data4_original = deepcopy(data4);

In [5]:
# Loop through each column of data1,2,3,4 and normalize using df_norm_params
for col in names(data1)
    μ = df_norm_params[1, col]  # mean
    σ = df_norm_params[2, col]  # std
    data1[!, col] .= (data1[!, col] .- μ) ./ σ
end

for col in names(data2)
    μ = df_norm_params[1, col]  # mean
    σ = df_norm_params[2, col]  # std
    data2[!, col] .= (data2[!, col] .- μ) ./ σ
end

for col in names(data3)
    μ = df_norm_params[1, col]  # mean
    σ = df_norm_params[2, col]  # std
    data3[!, col] .= (data3[!, col] .- μ) ./ σ
end

for col in names(data4)
    μ = df_norm_params[1, col]  # mean
    σ = df_norm_params[2, col]  # std
    data4[!, col] .= (data4[!, col] .- μ) ./ σ
end

## data1

In [6]:
# Cleaning up data1, to repeat for data2, 3, 4
using DataFrames, LinearAlgebra

# Step 1: Model parameters (above)

# Step 2: Extract features from DataFrame
features = Matrix(data1)

# Step 3: Compute v[i, k] = exp(β₀ₖ + sum_j βⱼₖ xⱼᵢ)
n = size(features, 1)
K = size(betas, 1)
v = Array{Float64}(undef, n, K)

for k in 1:K
    for i in 1:n
        utility = intercepts[k] + dot(betas[k, :], features[i, :])
        v[i, k] = exp(utility)
    end
end

# Price matrix
p = data1_original[:, :price_usd];

## Model
#### do not know customer segment type

$$
\begin{align*}
\max \quad & \sum_{k=1}^{2} \theta_k z_k \\
\text{s.t.} \quad 
& -M y_i \leq x_{ik} \leq M y_i, && \forall i \forall k \\
& -M(1 - y_i) + p_i - z_k \leq x_{ik} \leq p_i - z_k + M(1 - y_i) && \forall i, \forall k \\
& z_k \leq \sum_{i=1}^{n} x_{ik} v_{ik} && \forall k \\
& y_i \in \{0, 1\} && \forall i \\
& z_k \geq 0 && \forall k
\end{align*}
$$

In [7]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_segment_k = 2
num_hotels_i = nrow(data1)

#create variables
@variable(model, z[1:num_segment_k] >= 0)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i, 1:num_segment_k])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, sum(thetas[k] * z[k] for k in 1:num_segment_k))

# Set up constraints

# Constraint 1: -M y_i <= x_ik <= M y_i for all i, k
M = 1000
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * y[i] <= x[i, k])
    @constraint(model, x[i, k] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z_k <= x_ik <= p_i - z_k + M (1 - y_i) for all i, k
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * (1 - y[i]) + p[i] - z[k] <= x[i, k])
    @constraint(model, x[i, k] <= p[i] - z[k] + M * (1 - y[i]))
end

# Constraint 3: z_k <= Sum over i of x_ik v_ik for all k
 for k in 1:num_segment_k
    @constraint(model, z[k] <= sum(x[i, k] * v[i, k] for i in 1:num_hotels_i))
end


Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20


In [8]:
#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("data1 without knowing customer segment type")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))

selected_hotels = []

for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))

data1_S_rev = objective_value(model)
data1_S = copy(selected_hotels);

Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
data1 without knowing customer segment type
Optimal Objective Function value (Exp Revenue): 107.35217363043266
Hotels selected (indices): 0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26


## Model
#### know customer segment type

$$
\begin{align*}
\max z \\
\text{s.t.} \quad 
& -M y_i \leq x_{i} \leq M y_i, && \forall i \\
& -M(1 - y_i) + p_i - z \leq x_{i} \leq p_i - z + M(1 - y_i) && \forall i \\
& z \leq \sum_{i=1}^{n} x_{i} v_{i} \\
& y_i \in \{0, 1\} && \forall i \\
\end{align*}
$$

In [9]:
# Extract the two columns of v representing the two segments
v_1 = v[:, 1]
v_2 = v[:, 2];

In [10]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data1)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_1[i] for i in 1:num_hotels_i));

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20


In [11]:
#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S1 of data1")

println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))

selected_hotels = []

for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data1_S1_rev = objective_value(model)
data1_S1 = copy(selected_hotels);

Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S1 of data1
Optimal Objective Function value (Exp Revenue): 108.09118841600318
Hotels selected (indices): 0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26


In [12]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data1)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_2[i] for i in 1:num_hotels_i))

#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S2 of data1")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []

for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data1_S2_rev = objective_value(model)
data1_S2 = copy(selected_hotels);

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S2 of data1
Optimal Objective Function value (Exp Revenue): 106.82050963879034
Hotels selected (indices): 0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26


## data2

In [13]:
# Cleaning up data2
using DataFrames, LinearAlgebra

# Step 1: Model parameters (done above)

# Step 2: Extract features from DataFrame
features = Matrix(data2)

# Step 3: Compute v[i, k] = exp(β₀ₖ + sum_j βⱼₖ xⱼᵢ)
n = size(features, 1)
K = size(betas, 1)
v = Array{Float64}(undef, n, K)

for k in 1:K
    for i in 1:n
        utility = intercepts[k] + dot(betas[k, :], features[i, :])
        v[i, k] = exp(utility)
    end
end

# Price matrix
p = data2_original[:, :price_usd];

### data2 - do not know customer segment type

In [14]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_segment_k = 2
num_hotels_i = nrow(data2)

#create variables
@variable(model, z[1:num_segment_k] >= 0)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i, 1:num_segment_k])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, sum(thetas[k] * z[k] for k in 1:num_segment_k))

# Set up constraints

# Constraint 1: -M y_i <= x_ik <= M y_i for all i, k
M = maximum(p) + 10
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * y[i] <= x[i, k])
    @constraint(model, x[i, k] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z_k <= x_ik <= p_i - z_k + M (1 - y_i) for all i, k
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * (1 - y[i]) + p[i] - z[k] <= x[i, k])
    @constraint(model, x[i, k] <= p[i] - z[k] + M * (1 - y[i]))
end

# Constraint 3: z_k <= Sum over i of x_ik v_ik for all k
 for k in 1:num_segment_k
    @constraint(model, z[k] <= sum(x[i, k] * v[i, k] for i in 1:num_hotels_i))
end

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20


In [15]:
# Sort by price
sorted_indices = sortperm(data2[:, :price_usd], rev=true)  # highest price first

# Mark top 5 indices
top5_indices = sorted_indices[1:5]

# Set warm start values (without modifying data2)
for i in 1:num_hotels_i
    if i in top5_indices
        set_start_value(y[i], 1)
    else
        set_start_value(y[i], 0)
    end
end

set_optimizer_attribute(model, "TimeLimit", 180)       # Stop after 180 seconds
set_optimizer_attribute(model, "Presolve", 2)          # Use aggressive presolve
set_optimizer_attribute(model, "Cuts", 2)              # Use aggressive cuts

# Optimize the model
optimize!(model)

# Get status of optimization
status = termination_status(model)
raw = raw_status(model)
println("Status = ", raw)

# Print results based on status
println("data2 without knowing customer segment type")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data2_S_rev = objective_value(model)
data2_S = copy(selected_hotels);


Set parameter Cuts to value 2
Set parameter TimeLimit to value 180
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
data2 without knowing customer segment type
Optimal Objective Function value (Exp Revenue): 130.94868992632198
Hotels selected (indices): 0, 1, 6, 7, 8, 9, 10, 21, 23, 25


### data2 - know customer segment type

#### S1

In [16]:
# Extract the two columns of v representing the two segments
v_1 = v[:, 1]
v_2 = v[:, 2]

#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data2)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_1[i] for i in 1:num_hotels_i))

#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S1 of data2")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data2_S1_rev = objective_value(model)
data2_S1 = copy(selected_hotels);

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S1 of data2
Optimal Objective Function value (Exp Revenue): 132.4592170260214
Hotels selected (indices): 0, 1, 6, 7, 8, 9, 10, 21, 23, 25


#### S2

In [17]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data2)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_2[i] for i in 1:num_hotels_i))

#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S2 of data2")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data2_S2_rev = objective_value(model)
data2_S2 = copy(selected_hotels);

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S2 of data2
Optimal Objective Function value (Exp Revenue): 129.83950301336222
Hotels selected (indices): 0, 1, 6, 7, 8, 9, 10, 21, 23, 25


## data3

In [18]:
# Cleaning up data3
using DataFrames, LinearAlgebra

# Step 1: Model parameters (done above)

# Step 2: Extract features from DataFrame
features = Matrix(data3)

# Step 3: Compute v[i, k] = exp(β₀ₖ + sum_j βⱼₖ xⱼᵢ)
n = size(features, 1)
K = size(betas, 1)
v = Array{Float64}(undef, n, K)

for k in 1:K
    for i in 1:n
        utility = intercepts[k] + dot(betas[k, :], features[i, :])
        v[i, k] = exp(utility)
    end
end

# Price matrix
p = data3_original[:, :price_usd];

### data3 - do not know customer segment type

In [19]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_segment_k = 2
num_hotels_i = nrow(data3)

#create variables
@variable(model, z[1:num_segment_k] >= 0)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i, 1:num_segment_k])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, sum(thetas[k] * z[k] for k in 1:num_segment_k))

# Set up constraints

# Constraint 1: -M y_i <= x_ik <= M y_i for all i, k
M = maximum(p) + 10
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * y[i] <= x[i, k])
    @constraint(model, x[i, k] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z_k <= x_ik <= p_i - z_k + M (1 - y_i) for all i, k
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * (1 - y[i]) + p[i] - z[k] <= x[i, k])
    @constraint(model, x[i, k] <= p[i] - z[k] + M * (1 - y[i]))
end

# Constraint 3: z_k <= Sum over i of x_ik v_ik for all k
 for k in 1:num_segment_k
    @constraint(model, z[k] <= sum(x[i, k] * v[i, k] for i in 1:num_hotels_i))
end

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20


In [20]:
# Sort by price
sorted_indices = sortperm(data3[:, :price_usd], rev=true)  # highest price first

# Mark top 5 indices
top5_indices = sorted_indices[1:5]

# Set warm start values (without modifying data2)
for i in 1:num_hotels_i
    if i in top5_indices
        set_start_value(y[i], 1)
    else
        set_start_value(y[i], 0)
    end
end

set_optimizer_attribute(model, "TimeLimit", 180)       # Stop after 180 seconds
set_optimizer_attribute(model, "Presolve", 2)          # Use aggressive presolve
set_optimizer_attribute(model, "Cuts", 2)              # Use aggressive cuts

# Optimize the model
optimize!(model)

# Get status of optimization
status = termination_status(model)
raw = raw_status(model)
println("Status = ", raw)

# Print results based on status
println("data3 without knowing customer segment type")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data3_S_rev = objective_value(model)
data3_S = copy(selected_hotels);

Set parameter Cuts to value 2
Set parameter TimeLimit to value 180
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
data3 without knowing customer segment type
Optimal Objective Function value (Exp Revenue): 121.17314583189967
Hotels selected (indices): 0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24


### data3 - know customer segment type

#### S1

In [21]:
# Extract the two columns of v representing the two segments
v_1 = v[:, 1]
v_2 = v[:, 2]

#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data3)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_1[i] for i in 1:num_hotels_i))

#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S1 of data3")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data3_S1_rev = objective_value(model)
data3_S1 = copy(selected_hotels);

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S1 of data3
Optimal Objective Function value (Exp Revenue): 120.59367808510699
Hotels selected (indices): 0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24


#### S2

In [22]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data3)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_2[i] for i in 1:num_hotels_i))

#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S2 of data3")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data3_S2_rev = objective_value(model)
data3_S2 = copy(selected_hotels);

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S2 of data3
Optimal Objective Function value (Exp Revenue): 121.5986516314002
Hotels selected (indices): 0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24


## data4

In [23]:
# Cleaning up data4
using DataFrames, LinearAlgebra

# Step 1: Model parameters (done above)

# Step 2: Extract features from DataFrame
features = Matrix(data4)

# Step 3: Compute v[i, k] = exp(β₀ₖ + sum_j βⱼₖ xⱼᵢ)
n = size(features, 1)
K = size(betas, 1)
v = Array{Float64}(undef, n, K)

for k in 1:K
    for i in 1:n
        utility = intercepts[k] + dot(betas[k, :], features[i, :])
        v[i, k] = exp(utility)
    end
end

# Price matrix
p = data4_original[:, :price_usd];

### data4 - do not know customer segment type

In [24]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_segment_k = 2
num_hotels_i = nrow(data4)

#create variables
@variable(model, z[1:num_segment_k] >= 0)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i, 1:num_segment_k])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, sum(thetas[k] * z[k] for k in 1:num_segment_k))

# Set up constraints

# Constraint 1: -M y_i <= x_ik <= M y_i for all i, k
M = maximum(p) + 10
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * y[i] <= x[i, k])
    @constraint(model, x[i, k] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z_k <= x_ik <= p_i - z_k + M (1 - y_i) for all i, k
for i in 1:num_hotels_i, k in 1:num_segment_k
    @constraint(model, -M * (1 - y[i]) + p[i] - z[k] <= x[i, k])
    @constraint(model, x[i, k] <= p[i] - z[k] + M * (1 - y[i]))
end

# Constraint 3: z_k <= Sum over i of x_ik v_ik for all k
 for k in 1:num_segment_k
    @constraint(model, z[k] <= sum(x[i, k] * v[i, k] for i in 1:num_hotels_i))
end

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20


In [25]:
# Sort by price
sorted_indices = sortperm(data4[:, :price_usd], rev=true)  # highest price first

# Mark top 5 indices
top5_indices = sorted_indices[1:5]

# Set warm start values (without modifying data2)
for i in 1:num_hotels_i
    if i in top5_indices
        set_start_value(y[i], 1)
    else
        set_start_value(y[i], 0)
    end
end

set_optimizer_attribute(model, "TimeLimit", 180)       # Stop after 180 seconds
set_optimizer_attribute(model, "Presolve", 2)          # Use aggressive presolve
set_optimizer_attribute(model, "Cuts", 2)              # Use aggressive cuts

# Optimize the model
optimize!(model)

# Get status of optimization
status = termination_status(model)
raw = raw_status(model)
println("Status = ", raw)

# Print results based on status
println("data4 without knowing customer segment type")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data4_S_rev = objective_value(model)
data4_S = copy(selected_hotels);

Set parameter Cuts to value 2
Set parameter TimeLimit to value 180
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
data4 without knowing customer segment type
Optimal Objective Function value (Exp Revenue): 97.26345071969826
Hotels selected (indices): 3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26


### data4 - know customer segment type

#### S1

In [26]:
# Extract the two columns of v representing the two segments
v_1 = v[:, 1]
v_2 = v[:, 2]

#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data4)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_1[i] for i in 1:num_hotels_i))

#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S1 of data4")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data4_S1_rev = objective_value(model)
data4_S1 = copy(selected_hotels);

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S1 of data4
Optimal Objective Function value (Exp Revenue): 98.16774262859315
Hotels selected (indices): 3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26


#### S2

In [27]:
#Building the JuMP model
using JuMP, Gurobi, LinearAlgebra

#Initialize the model with a given optimizer, say Gurobi
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "OutputFlag", 0)

# Define the variables
num_hotels_i = nrow(data4)

#create variables
@variable(model, z)
@variable(model, y[1:num_hotels_i], Bin)
@variable(model, x[1:num_hotels_i])

#Set the objective. The arguments are the model object, the objective sense and its expression.
@objective(model, Max, z)

# Set up constraints

# Constraint 1: -M y_i <= x_i <= M y_i for all i
M = 1000
for i in 1:num_hotels_i
    @constraint(model, -M * y[i] <= x[i])
    @constraint(model, x[i] <= M * y[i])
end

# Constraint 2: -M (1-y_i) + p_i - z <= x_i <= p_i - z + M (1 - y_i) for all i
for i in 1:num_hotels_i
    @constraint(model, -M * (1 - y[i]) + p[i] - z <= x[i])
    @constraint(model, x[i] <= p[i] - z + M * (1 - y[i]))
end

# Constraint 3: z <= Sum over i of x_i v_i
@constraint(model, z <= sum(x[i] * v_2[i] for i in 1:num_hotels_i))

#optimize the model
optimize!(model)

#get the status of the optimization
#either
println("Status = ", raw_status(model))

#print the solution: value and variables
println("S2 of data4")
println("Optimal Objective Function value (Exp Revenue): ", objective_value(model))
selected_hotels = []
for i in 1:num_hotels_i
    if value(y[i]) == 1
        push!(selected_hotels, string(i - 1))
    end
end
println("Hotels selected (indices): ", join(selected_hotels, ", "))
data4_S2_rev = objective_value(model)
data4_S2 = copy(selected_hotels);

Set parameter Username
Academic license - for non-commercial use only - expires 2025-10-20
Status = Model was solved to optimality (subject to tolerances), and an optimal solution is available.
S2 of data4
Optimal Objective Function value (Exp Revenue): 96.59942506200478
Hotels selected (indices): 3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26


### Summary of all revenues and hotels selected

In [28]:
println(join(data1_S, ", "))
println(join(data1_S1, ", "))
println(join(data1_S2, ", "))

println(join(data2_S, ", "))
println(join(data2_S1, ", "))
println(join(data2_S2, ", "))

println(join(data3_S, ", "))
println(join(data3_S1, ", "))
println(join(data3_S2, ", "))

println(join(data4_S, ", "))
println(join(data4_S1, ", "))
println(join(data4_S2, ", "))

0, 1, 2, 3, 4, 5, 6, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26
0, 1, 2, 3, 4, 5, 6, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26
0, 1, 2, 3, 4, 5, 6, 9, 12, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 26
0, 1, 6, 7, 8, 9, 10, 21, 23, 25
0, 1, 6, 7, 8, 9, 10, 21, 23, 25
0, 1, 6, 7, 8, 9, 10, 21, 23, 25
0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24
0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24
0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 14, 15, 16, 18, 19, 23, 24
3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26
3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26
3, 4, 6, 8, 10, 15, 18, 19, 20, 21, 26


In [29]:
using Printf

@printf("%.2f\n", data1_S_rev)
@printf("%.2f\n", data1_S1_rev)
@printf("%.2f\n", data1_S2_rev)

@printf("%.2f\n", data2_S_rev)
@printf("%.2f\n", data2_S1_rev)
@printf("%.2f\n", data2_S2_rev)

@printf("%.2f\n", data3_S_rev)
@printf("%.2f\n", data3_S1_rev)
@printf("%.2f\n", data3_S2_rev)

@printf("%.2f\n", data4_S_rev)
@printf("%.2f\n", data4_S1_rev)
@printf("%.2f\n", data4_S2_rev)


107.35
108.09
106.82
130.95
132.46
129.84
121.17
120.59
121.60
97.26
98.17
96.60
